In [ ]:
!pip install findspark

In [ ]:
!pip install pyspark

# Анализ и очистка датасета мошеннических транзакций

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.storagelevel import StorageLevel

spark = (SparkSession.builder
         .appName("FraudCleaning")
         .getOrCreate())

spark.conf.set("spark.sql.adaptive.enabled", "true")


## Загрузка данных

In [ ]:
schema = '''
transaction_id LONG,
tx_datetime STRING,
customer_id LONG,
terminal_id LONG,
tx_amount DOUBLE,
tx_time_seconds LONG,
tx_time_days INT,
tx_fraud INT,
tx_fraud_scenario INT
'''

df = (spark.read
      .option("header", "false")
      .schema(schema)
      .csv("hdfs://rc1a-dataproc-m-lj7q01ef2tjqlm9m.mdb.yandexcloud.net:8020/user/data/fraud/*.txt")
      .repartition(64))

df.persist(StorageLevel.MEMORY_AND_DISK)
df.count()


## Анализ качества данных

In [ ]:
null_stats = df.select([sum(col(c).isNull().cast("int")).alias(c) for c in df.columns])
null_stats.show(truncate=False)

df_dates = df.withColumn("parsed_datetime", to_timestamp("tx_datetime"))

bad_dates = df_dates.filter(col("parsed_datetime").isNull()).count()
negative_amounts = df.filter(col("tx_amount") <= 0).count()
duplicates = df.groupBy("transaction_id").count().filter(col("count") > 1).count()
invalid_fraud = df.filter(~col("tx_fraud").isin(0,1)).count()

print("bad_dates =", bad_dates)
print("negative_amounts =", negative_amounts)
print("duplicates =", duplicates)
print("invalid_fraud =", invalid_fraud)


## Выбросы

In [ ]:
stats = df.selectExpr(
    "percentile_approx(tx_amount,0.25) q1",
    "percentile_approx(tx_amount,0.75) q3"
).collect()[0]

q1, q3 = stats["q1"], stats["q3"]
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
print(lower, upper)


## Очистка данных

In [ ]:
median_amount = df.approxQuantile("tx_amount",[0.5],0.01)[0]

clean_df = (
    df_dates
    .filter(col("parsed_datetime").isNotNull())
    .filter(col("tx_fraud").isin(0,1))
    .filter(col("customer_id") >= 0)
    .filter(col("terminal_id") >= 0)
    .dropDuplicates(["transaction_id"])
    .withColumn(
        "tx_amount",
        when(col("tx_amount") <= 0, median_amount)
        .when(col("tx_amount") < lower, lower)
        .when(col("tx_amount") > upper, upper)
        .otherwise(col("tx_amount"))
    )
    .drop("parsed_datetime")
)


## Сохранение в Parquet

In [ ]:
clean_df.write \
    .mode("overwrite") \
    .option("compression","snappy") \
    .parquet("s3a://spark-bucket-ek/fraud_clean_parquet")
